# Rotation estimation by feature matching

We performed 5 consecutive acquisitions of the same object (without moving it) with the _Plant Imager_.
The first image of the first dataset present a different pose than the other four.

We would like to estimate the difference in terms of **relative camera poses**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.feature import match_descriptors, ORB, plot_matches
from skimage.measure import ransac
from skimage.transform import AffineTransform

## Load the images data

In [ ]:
root_path = Path("/data/ROMI/2022.10.10_Hardware_repeatability/Scans")
img_left = imread(root_path.joinpath('Sangoku_40_1/images/00000_rgb.jpg'))
img_right = imread(root_path.joinpath('Sangoku_40_2/images/00000_rgb.jpg'))

In [ ]:
img_left, img_right = map(rgb2gray, (img_left, img_right))

In [ ]:
nr, nc = img_left.shape

In [ ]:
# build an RGB image with the unregistered sequence
unreg_im = np.zeros((nr, nc, 3))
unreg_im[..., 0] = img_left
unreg_im[..., 1] = img_right
unreg_im[..., 2] = img_right

In [ ]:
plt.imshow(unreg_im)

## Find sparse feature correspondences between left and right image.

In [ ]:
descriptor_extractor = ORB()

descriptor_extractor.detect_and_extract(img_left)
keypoints_left = descriptor_extractor.keypoints
descriptors_left = descriptor_extractor.descriptors

descriptor_extractor.detect_and_extract(img_right)
keypoints_right = descriptor_extractor.keypoints
descriptors_right = descriptor_extractor.descriptors

In [ ]:
matches = match_descriptors(descriptors_left, descriptors_right,
                            cross_check=True)

print(f'Number of matches: {matches.shape[0]}')

In [ ]:
# Visualize the results.
fig, ax = plt.subplots()
plt.gray()
plot_matches(ax, img_left, img_right, keypoints_left, keypoints_right,
             matches[inliers], only_matches=True)
ax.axis("off")
ax.set_title("Inlier correspondences")
plt.show()

## Estimate the epipolar geometry between the left and right image.

In [ ]:
model, inliers = ransac((keypoints_left[matches[:, 0]],
                         keypoints_right[matches[:, 1]]),
                        AffineTransform, min_samples=8,
                        residual_threshold=1, max_trials=5000)

In [ ]:
model.translation